# ADT→platinum vs ADT→NEPC endpoint comparison

Read-only. This notebook compares the two survival runs that share the ADT
anchor (first ADT exposure = time 0) but model different events:

| run | tree | event | duration |
|---|---|---|---|
| platinum | `local_runs_adt/` | `PLATINUM` | `t_platinum` |
| NEPC | `local_runs_adt_nepc/` | `NEPC` | `t_nepc` |

It refits nothing — run `01`/`02`/`03` once per endpoint first (see their
`ENDPOINT` / `OUTPUT_SUFFIX` parameter cells).

### Two things to keep in view while reading

**These are not the same patients.** The NEPC build is gated on `t_nepc > 0`,
which excludes *prevalent* NEPC (diagnosed at or before the landmark) to make
it an incident endpoint. Section 1 quantifies that difference. Metric
differences between the two runs therefore confound *event* with *cohort*.

**The NEPC labels here are the strict definition.** They come from the
veto-gated `nepc_diagnosis` pipeline, which answers only "does the record
state an NEPC diagnosis, and when?". That is deliberately narrower than the
"any NE feature → NEPC" rule used by the binary classifier in the Figure 2
enrichment analysis. The two are **not** interchangeable, and any text drawn
from this notebook should name which definition it means.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Override if the COMPASS output tree is mounted elsewhere.
DATA_ROOT = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS")
SURVIVAL_ROOT = DATA_ROOT / "survival_analysis"

ARM = "adt"
LANDMARKS = (0, 90, 180)
ID_COL = "DFCI_MRN"

# (endpoint label) -> (results tree, prediction-inputs tree)
TREES = {
    "platinum": {
        "results": SURVIVAL_ROOT / f"local_runs_{ARM}",
        "inputs": SURVIVAL_ROOT / f"prediction_inputs_{ARM}",
        "event_col": "PLATINUM",
        "duration_col": "t_platinum",
    },
    "nepc": {
        "results": SURVIVAL_ROOT / f"local_runs_{ARM}_nepc",
        "inputs": SURVIVAL_ROOT / f"prediction_inputs_{ARM}_nepc",
        "event_col": "NEPC",
        "duration_col": "t_nepc",
    },
}

# Stage 1 cohort, which carries both endpoints' columns before any landmark gating.
COHORT_PATH = DATA_ROOT / f"prostate_{ARM}_survival_cohort_{ARM}.csv"


def normalize_id(series):
    """String-normalize MRNs so int/float round-trips don't break merges."""
    return pd.to_numeric(series, errors="coerce").astype("Int64").astype("string")


# Collect missing artifacts and report them rather than crashing: a partially
# complete run should still show whatever sections it can.
MISSING: list[str] = []


def require(path: Path, what: str) -> bool:
    if path.exists():
        return True
    MISSING.append(f"{what}: {path}")
    return False


print(f"data root: {DATA_ROOT}")
for endpoint, spec in TREES.items():
    for kind in ("results", "inputs"):
        mark = "ok     " if spec[kind].exists() else "MISSING"
        print(f"  {mark} {endpoint:<9s} {kind:<7s} {spec[kind]}")

## 1. Cohort and event counts

**The most important table in this notebook.** If the NEPC event count is very
small, every model comparison below is underpowered and should not be read as
"NEPC is harder to predict than platinum".

The Stage 1 row is the cohort *before* landmark gating; the per-landmark rows
are what each model actually fit.

In [ ]:
cohort = None
if require(COHORT_PATH, "Stage 1 survival cohort"):
    cohort = pd.read_csv(COHORT_PATH, low_memory=False)
    cohort[ID_COL] = normalize_id(cohort[ID_COL])

    stage1_rows = []
    for endpoint, spec in TREES.items():
        event_col = spec["event_col"]
        if event_col not in cohort.columns:
            stage1_rows.append({
                "endpoint": endpoint, "n_patients": len(cohort),
                "n_events": np.nan, "event_rate_pct": np.nan,
                "note": f"{event_col} absent from the Stage 1 cohort",
            })
            continue
        events = pd.to_numeric(cohort[event_col], errors="coerce").fillna(0)
        tt_col = f"TT_{event_col}"
        # Prevalent = event at or before the anchor; the incident filter drops these.
        n_prevalent = np.nan
        if tt_col in cohort.columns:
            tt = pd.to_numeric(cohort[tt_col], errors="coerce")
            n_prevalent = int((events.eq(1) & tt.le(0)).sum())
        stage1_rows.append({
            "endpoint": endpoint,
            "n_patients": len(cohort),
            "n_events": int(events.eq(1).sum()),
            "event_rate_pct": 100 * events.eq(1).mean(),
            "n_prevalent_at_anchor": n_prevalent,
            "note": "",
        })

    print("Stage 1 cohort (before any landmark gating):")
    display(pd.DataFrame(stage1_rows).round(2))

In [ ]:
landmark_rows = []
for endpoint, spec in TREES.items():
    for landmark in LANDMARKS:
        path = spec["inputs"] / f"aggregated_landmark{landmark}.csv"
        base = {"endpoint": endpoint, "landmark_days": landmark}
        if not require(path, f"{endpoint} aggregated landmark {landmark}"):
            landmark_rows.append({**base, "status": "missing"})
            continue
        agg = pd.read_csv(path, low_memory=False)
        event_col, duration_col = spec["event_col"], spec["duration_col"]
        if event_col not in agg.columns:
            landmark_rows.append({**base, "status": f"no {event_col} column"})
            continue
        events = pd.to_numeric(agg[event_col], errors="coerce").fillna(0)
        durations = pd.to_numeric(agg.get(duration_col), errors="coerce")
        landmark_rows.append({
            **base,
            "n_patients": len(agg),
            "n_events": int(events.eq(1).sum()),
            "event_rate_pct": 100 * events.eq(1).mean(),
            "median_days_to_event": durations.loc[events.eq(1)].median(),
            "median_days_followup": durations.median(),
            "status": "ok",
        })

cohort_table = pd.DataFrame(landmark_rows)
display(cohort_table.round(1))

# Loudly flag a thin endpoint: rules of thumb put stable multivariable Cox at
# roughly 10 events per covariate, so a double-digit event count cannot support
# the full lab feature set no matter what the C-index says.
ok = cohort_table.query("status == 'ok'") if "status" in cohort_table else cohort_table
for _, row in ok.iterrows():
    if row["n_events"] < 50:
        print(
            f"!! {row['endpoint']} @ landmark {row['landmark_days']}d has only "
            f"{int(row['n_events'])} events — treat every model metric below as "
            "underpowered, not as evidence about predictability."
        )

### Cohort overlap

How much of the difference between the two runs is *cohort* rather than
*event*. Patients in the platinum cohort but not the NEPC cohort were dropped
by the incident-NEPC gate.

In [ ]:
overlap_rows = []
for landmark in LANDMARKS:
    paths = {
        ep: TREES[ep]["inputs"] / f"aggregated_landmark{landmark}.csv"
        for ep in TREES
    }
    if not all(p.exists() for p in paths.values()):
        continue
    ids = {}
    for ep, p in paths.items():
        frame = pd.read_csv(p, usecols=[ID_COL], low_memory=False)
        ids[ep] = set(normalize_id(frame[ID_COL]).dropna())
    plat, nepc = ids["platinum"], ids["nepc"]
    overlap_rows.append({
        "landmark_days": landmark,
        "n_platinum_cohort": len(plat),
        "n_nepc_cohort": len(nepc),
        "n_shared": len(plat & nepc),
        "n_platinum_only": len(plat - nepc),
        "n_nepc_only": len(nepc - plat),
        "pct_of_platinum_retained": 100 * len(plat & nepc) / len(plat) if plat else np.nan,
    })

if overlap_rows:
    display(pd.DataFrame(overlap_rows).round(1))
else:
    print("Both trees are needed for the overlap table; see the missing-artifact list.")

## 2. NEPC event provenance

This is the sensitivity analysis the retained provenance flags exist for.

- `date_source` — `stated` means the note gave an explicit diagnosis date.
  **`note_date` means the date is earliest *documentation*, not onset**, which
  biases `t_nepc` late by an unknown amount.
- `date_precision` — `month`/`year` dates were normalized to the 1st, so they
  carry up to a year of error.
- `label_source` — `auto_negative_no_evidence` patients were never seen by an
  LLM. They are legitimate censored observations but are not adjudicated
  negatives.

If a large share of events are `note_date` or coarse-precision, re-run the
comparison restricted to `stated`/`day` events before drawing conclusions.

In [ ]:
PROVENANCE_COLS = ["NEPC_DATE_SOURCE", "NEPC_DATE_PRECISION", "NEPC_LABEL_SOURCE"]

if cohort is not None and "NEPC" in cohort.columns:
    positives = cohort.loc[pd.to_numeric(cohort["NEPC"], errors="coerce").eq(1)]
    print(f"NEPC positives in the Stage 1 cohort: {len(positives):,}")
    for col in PROVENANCE_COLS:
        if col not in cohort.columns:
            print(f"  ({col} not carried in this cohort build)")
            continue
        counts = positives[col].fillna("(null)").value_counts()
        breakdown = pd.DataFrame({
            "n": counts,
            "percent_of_events": 100 * counts / max(len(positives), 1),
        })
        print(f"\n{col}:")
        display(breakdown.round(1))

    # label_source across everyone, not just events: how much of the censored
    # mass is auto-negative (never adjudicated) rather than LLM-confirmed.
    if "NEPC_LABEL_SOURCE" in cohort.columns:
        print("\nlabel_source across the whole cohort (censored included):")
        display(
            cohort["NEPC_LABEL_SOURCE"].fillna("(null)").value_counts().to_frame("n")
        )
else:
    print("No NEPC columns in the Stage 1 cohort — rebuild it with --nepc-labels.")

## 3. Held-out model performance

Test-set metrics only, never cross-validation. Both trees emit the same schema
via `compass_pipeline.summarize_outputs`, so they concatenate directly.

Read across the `endpoint` column with section 1 in hand: a lower NEPC C-index
on a few dozen events is a statement about power, not about biology.

In [ ]:
import sys
sys.path.insert(0, ".")
import compass_pipeline as cp

summaries = []
for endpoint, spec in TREES.items():
    # summarize_outputs reads run["output_dir"]/run["endpoint"]; build the run
    # dict directly so this notebook never touches make_runs' mkdir side effects.
    run = {
        "label": ARM,
        "output_dir": spec["results"],
        "landmarks": list(LANDMARKS),
        "endpoint": endpoint,
    }
    if not spec["results"].exists():
        MISSING.append(f"{endpoint} results tree: {spec['results']}")
        continue
    summaries.append(cp.summarize_outputs(run))

if summaries:
    performance = pd.concat(summaries, ignore_index=True)
    display(performance.round(3))
else:
    performance = pd.DataFrame()
    print("No results trees found; see the missing-artifact list at the end.")

In [ ]:
if not performance.empty:
    ok_perf = performance.loc[performance["status"].eq("ok")]
    for metric in ("c_index", "mean_auc_t", "integrated_brier"):
        if ok_perf.empty:
            break
        pivot = ok_perf.pivot_table(
            index=["model", "config", "landmark"],
            columns="endpoint",
            values=metric,
        )
        if {"platinum", "nepc"}.issubset(pivot.columns):
            pivot["nepc - platinum"] = pivot["nepc"] - pivot["platinum"]
        print(f"\n=== {metric} ===")
        display(pivot.round(3))

## 4. Univariate association overlap

Which lab associations are shared between the two endpoints and which are
endpoint-specific, joined on `(feature, landmark_days)` with hazard ratios side
by side.

A feature nominally significant for platinum but not NEPC is *not* evidence of
a real difference when the NEPC arm has far fewer events — the confidence
intervals do that work, so they are shown too.

In [ ]:
HR_COL = "hazard_ratio_per_sd"

univariate_frames = []
for endpoint, spec in TREES.items():
    for landmark in LANDMARKS:
        path = (
            spec["results"] / "cox" / f"landmark_{landmark}" / "both"
            / "cox_agg_univariate_nobs_adjusted.csv"
        )
        if not require(path, f"{endpoint} univariate landmark {landmark}"):
            continue
        frame = pd.read_csv(path)
        frame = frame.loc[
            frame["endpoint"].astype(str).str.lower().eq(endpoint)
        ].copy()
        frame["landmark_days"] = landmark
        frame["endpoint"] = endpoint
        univariate_frames.append(frame)

if univariate_frames:
    univariate = pd.concat(univariate_frames, ignore_index=True)
    keep = [
        "endpoint", "landmark_days", "feature", "lab_name", "feature_stat",
        "n_patients_used", "n_events_used", HR_COL, "ci_lower", "ci_upper",
        "p_value", "q_value",
    ]
    univariate = univariate[[c for c in keep if c in univariate.columns]]

    wide = univariate.pivot_table(
        index=["landmark_days", "feature"],
        columns="endpoint",
        values=[HR_COL, "p_value", "n_events_used"],
    )
    wide.columns = [f"{metric}_{ep}" for metric, ep in wide.columns]

    p_plat, p_nepc = "p_value_platinum", "p_value_nepc"
    if p_plat in wide.columns and p_nepc in wide.columns:
        sig_plat = wide[p_plat].lt(0.05)
        sig_nepc = wide[p_nepc].lt(0.05)
        wide["shared_nominal"] = sig_plat & sig_nepc
        wide["platinum_only"] = sig_plat & ~sig_nepc
        wide["nepc_only"] = sig_nepc & ~sig_plat
        print(
            f"nominally significant (p<0.05) in both: {int(wide['shared_nominal'].sum())}; "
            f"platinum only: {int(wide['platinum_only'].sum())}; "
            f"NEPC only: {int(wide['nepc_only'].sum())}"
        )
        display(
            wide.loc[wide["shared_nominal"] | wide["platinum_only"] | wide["nepc_only"]]
            .sort_values(p_plat)
            .round(3)
        )
    else:
        display(wide.round(3))
else:
    univariate = pd.DataFrame()
    print("No univariate results found for either endpoint.")

## 5. Kaplan–Meier: event-free survival from the ADT anchor

Platinum-free and NEPC-free survival on the same axis. Each curve is drawn on
its own cohort (see section 1), so this is a descriptive side-by-side, not a
competing-risks decomposition — a patient can appear in both.

In [ ]:
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter

KM_LANDMARK = 0

fig, ax = plt.subplots(figsize=(7, 5))
drew_any = False
for endpoint, spec in TREES.items():
    path = spec["inputs"] / f"aggregated_landmark{KM_LANDMARK}.csv"
    if not path.exists():
        continue
    agg = pd.read_csv(path, low_memory=False)
    event_col, duration_col = spec["event_col"], spec["duration_col"]
    if event_col not in agg.columns or duration_col not in agg.columns:
        continue
    durations = pd.to_numeric(agg[duration_col], errors="coerce")
    events = pd.to_numeric(agg[event_col], errors="coerce").fillna(0)
    valid = durations.notna() & durations.gt(0)
    if not valid.any():
        continue
    kmf = KaplanMeierFitter()
    kmf.fit(
        durations.loc[valid],
        events.loc[valid],
        label=f"{endpoint} (n={int(valid.sum())}, events={int(events.loc[valid].eq(1).sum())})",
    )
    kmf.plot_survival_function(ax=ax, ci_show=True)
    drew_any = True

if drew_any:
    ax.set_xlabel("Days from ADT start")
    ax.set_ylabel("Event-free probability")
    ax.set_title(f"Event-free survival from the ADT anchor (landmark +{KM_LANDMARK}d)")
    ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    plt.close(fig)
    print("Neither endpoint had a usable landmark cohort for the KM plot.")

## Missing artifacts

In [ ]:
if MISSING:
    print(f"{len(MISSING)} artifact(s) were not found:")
    for item in dict.fromkeys(MISSING):
        print(f"  {item}")
    print(
        "\nRun 01/02/03 for the corresponding endpoint "
        "(ENDPOINT / OUTPUT_SUFFIX in their parameter cells)."
    )
else:
    print("All expected artifacts were found.")